# Course tools and Python foundations

**MECE 4520 · Fall 2026 · asynchronous foundations**

<a target="_blank" href="https://colab.research.google.com/github/changyaochen/MECE4520/blob/master/site/foundations/01-course-tools.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This notebook introduces the small set of Python tools used throughout the course. It is not a comprehensive Python tutorial: the goal is to be comfortable loading, inspecting, selecting, filtering, and plotting engineering data.

**Learning objectives**

- Use NumPy arrays for basic numerical calculations.
- Load a public dataset into a Pandas DataFrame.
- Select columns and filter rows using clear, readable code.
- Make and interpret a simple engineering-data plot.

## 1. Set up the notebook

Colab already includes NumPy, Pandas, and Matplotlib. The next cell installs `ucimlrepo`, a small package that downloads the dataset directly from the UCI Machine Learning Repository.

In [ ]:
%pip -q install ucimlrepo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ucimlrepo import fetch_ucirepo

plt.style.use("seaborn-v0_8-whitegrid")

## 2. Numerical arrays

A NumPy array stores a collection of numerical values. Array operations act element-by-element, which makes calculations compact and readable.

In [ ]:
temperatures_c = np.array([15.2, 18.7, 21.4, 24.1])
temperatures_k = temperatures_c + 273.15

print("Temperatures in K:", temperatures_k)
print(f"Mean temperature: {temperatures_c.mean():.1f} °C")
print(f"Range: {temperatures_c.min():.1f}–{temperatures_c.max():.1f} °C")

The array is useful for a single quantity observed repeatedly. A table with many named variables is better represented by a Pandas **DataFrame**.

## 3. Load the course dataset

The Gas Turbine CO and NOx Emissions dataset contains hourly sensor measurements and emissions outcomes. `features` and `targets` are returned separately; combining them gives one analysis table.

In [ ]:
gas_turbine = fetch_ucirepo(id=551)
features = gas_turbine.data.features.copy()
targets = gas_turbine.data.targets.copy()
data = pd.concat([features, targets], axis=1)

print(f"Rows: {len(data):,}")
print(f"Columns: {data.shape[1]}")
data.head()

A good first habit is to inspect the column names and units before modeling. The dictionary below records the variables we will use most often.

In [ ]:
variable_notes = {
    "AT": "ambient temperature (°C)",
    "AP": "ambient pressure (mbar)",
    "AH": "ambient humidity (%)",
    "TIT": "turbine inlet temperature (°C)",
    "TEY": "turbine energy yield (MWh)",
    "CO": "carbon monoxide emissions (mg/m³)",
    "NOX": "nitrogen oxides emissions (mg/m³)",
}

pd.Series(variable_notes, name="meaning")

## 4. Select and filter data

Use a list of column names to select several variables. Use a Boolean condition inside square brackets to retain rows meeting a condition.

In [ ]:
selected = data[["AT", "TIT", "TEY", "CO", "NOX"]]
warm_conditions = selected[selected["AT"] > selected["AT"].median()]

print("Selected variables:")
display(selected.head())
print(f"Rows above the median ambient temperature: {len(warm_conditions):,}")
warm_conditions.describe().loc[["mean", "std", "min", "max"]]

## 5. Make a first plot

A scatter plot helps us inspect whether two variables move together. Each point below is one hourly observation. We draw only a random subset so the plot remains readable.

In [ ]:
plot_data = data.sample(n=2_000, random_state=4520)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(plot_data["TIT"], plot_data["NOX"], alpha=0.25, s=12)
ax.set(
    xlabel="Turbine inlet temperature, TIT (°C)",
    ylabel="NOx emissions (mg/m³)",
    title="A first view of turbine inlet temperature and NOx emissions",
)
plt.show()

## Check-in

1. Change the plotting code to use `AT` or `TEY` on the horizontal axis. What pattern do you see?
2. What is one question you would ask before interpreting this plot as a causal relationship?
3. In one sentence, explain the difference between a NumPy array and a Pandas DataFrame.

Keep your answers in your own notes. We will build on this dataset in the next notebook.